# 03 · Visualize

讀 `data/processed/village_to_nearest_library_{profile}.csv` + 里界 GeoJSON，輸出三組地圖：

1. **行車時間**：`tainan_library_drive_time_*` (driving 來源)
2. **步行時間**：`tainan_library_walk_time_*` (walking 來源)
3. **行車距離 km**：`tainan_library_drive_dist_*` (driving 來源的 OSRM 道路距離)

每組產出靜態 PNG 和互動 HTML 兩種。

如果某個 profile 的 CSV 不存在（例如還沒跑 walking），對應的 map 會 skip。


In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from lib.colors import DRIVING, WALKING, DISTANCE

RAW_DIR = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"
OUTPUT_DIR = ROOT / "output" / "maps"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Try to use a CJK font for matplotlib titles
try:
    plt.rcParams["font.sans-serif"] = ["PingFang TC", "Heiti TC", "Arial Unicode MS"]
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

villages = gpd.read_file(RAW_DIR / "tainan_villages.geojson")
libs = pd.read_csv(RAW_DIR / "tainan_libraries.csv")
libs_gdf = gpd.GeoDataFrame(libs, geometry=gpd.points_from_xy(libs.lon, libs.lat), crs="EPSG:4326")

def load_profile(profile: str) -> pd.DataFrame | None:
    """Load the village→library CSV for a given OSRM profile. None if missing."""
    fp = PROC_DIR / f"village_to_nearest_library_{profile}.csv"
    if not fp.exists():
        print(f"⚠️  {fp.name} not found — skipping")
        return None
    return pd.read_csv(fp, dtype={"village_id": str})

driving_df = load_profile("driving")
walking_df = load_profile("walking")
print(f"driving rows: {len(driving_df) if driving_df is not None else 0}")
print(f"walking rows: {len(walking_df) if walking_df is not None else 0}")

In [ ]:
def render_static_map(merged, scale, value_col, value_unit, title, output_path):
    """Render a choropleth PNG for the given scale (DRIVING / WALKING / DISTANCE)."""
    cmap = ListedColormap(scale.colors)
    bounds = [0, *scale.cuts, 1e9]
    norm = BoundaryNorm(bounds, cmap.N)

    fig, ax = plt.subplots(figsize=(12, 14), dpi=150)

    merged.plot(
        column=value_col, cmap=cmap, norm=norm,
        edgecolor="white", linewidth=0.15, ax=ax,
        missing_kwds={"color": "lightgray", "label": "no data"},
    )

    # District boundaries (thicker)
    districts = merged.dissolve(by="district", as_index=False)
    districts.boundary.plot(ax=ax, color="black", linewidth=0.6)

    # Libraries as markers
    libs_gdf.plot(ax=ax, marker="P", color="black", markersize=40,
                  edgecolor="white", linewidth=0.5)

    # Legend
    handles = [Patch(facecolor=c, edgecolor="white",
                     label=f"{lab} {value_unit}")
               for c, lab in zip(scale.colors, scale.labels)]
    handles.append(Patch(facecolor="lightgray", edgecolor="white", label="無資料"))
    ax.legend(handles=handles, title=title, loc="lower left", fontsize=9)

    ax.set_title(f"台南市各里：{title}", fontsize=14, pad=10)
    ax.set_axis_off()
    ax.set_aspect("equal")

    fig.savefig(output_path, bbox_inches="tight", dpi=150)
    plt.show()
    plt.close(fig)
    print(f"✅ Saved {output_path}")

In [ ]:
configs = []
if driving_df is not None:
    configs.append(("driving", driving_df, DRIVING, "time_min", "分鐘",
                    "到最近市立圖書館行車時間",
                    OUTPUT_DIR / "tainan_library_drive_time_static.png"))
    configs.append(("driving", driving_df, DISTANCE, "distance_km", "km",
                    "到最近市立圖書館行車距離",
                    OUTPUT_DIR / "tainan_library_drive_dist_static.png"))
if walking_df is not None:
    configs.append(("walking", walking_df, WALKING, "time_min", "分鐘",
                    "到最近市立圖書館步行時間",
                    OUTPUT_DIR / "tainan_library_walk_time_static.png"))

for profile, df, scale, col, unit, title, path in configs:
    merged = villages.merge(
        df[["village_id", col, "nearest_library", "method"]],
        on="village_id", how="left",
    )
    print(f"--- {title} ({profile}) ---")
    print(f"  range: {merged[col].min():.2f} – {merged[col].max():.2f} {unit}")
    render_static_map(merged, scale, col, unit, title, path)
    print()

In [ ]:
import folium
from folium.features import GeoJsonTooltip

def render_interactive_map(merged, scale, value_col, value_unit, title, output_path):
    merged = merged.copy()
    merged["fill_color"] = merged[value_col].apply(
        lambda v: scale.color(v) if pd.notna(v) else "#cccccc"
    )
    merged["value_str"] = merged[value_col].apply(
        lambda v: f"{v:.1f}" if pd.notna(v) else "N/A"
    )

    centroid = merged.geometry.union_all().centroid
    m = folium.Map(location=[centroid.y, centroid.x], zoom_start=11, tiles="cartodbpositron")

    folium.GeoJson(
        merged.to_json(),
        name=title,
        style_function=lambda feat: {
            "fillColor": feat["properties"]["fill_color"],
            "color": "white", "weight": 0.3, "fillOpacity": 0.75,
        },
        tooltip=GeoJsonTooltip(
            fields=["village_name", "district", "nearest_library", "value_str", "method"],
            aliases=["里", "區", "最近圖書館", f"值 ({value_unit})", "計算方式"],
            sticky=True,
        ),
    ).add_to(m)

    for _, lib in libs.iterrows():
        folium.Marker(
            location=[lib["lat"], lib["lon"]],
            popup=folium.Popup(f"<b>{lib['name']}</b><br>{lib['district']}<br>{lib['address']}", max_width=300),
            icon=folium.Icon(color="black", icon="book", prefix="fa"),
        ).add_to(m)

    legend = """<div style="position: fixed; bottom: 20px; left: 20px; z-index: 9999;
        background: white; padding: 10px; border: 1px solid #999;
        font-family: sans-serif; font-size: 12px;"><b>""" + title + f" ({value_unit})</b><br>"
    legend += "".join(
        f'<div><span style="display:inline-block;width:14px;height:14px;background:{c};margin-right:6px;"></span>{lab}</div>'
        for c, lab in zip(scale.colors, scale.labels)
    )
    legend += "</div>"
    m.get_root().html.add_child(folium.Element(legend))

    m.save(str(output_path))
    print(f"✅ Saved {output_path}")
    return m


# Render all interactive maps
last_map = None
interactive_paths = [
    ("driving", driving_df, DRIVING, "time_min", "分鐘", "到最近圖書館行車時間",
     OUTPUT_DIR / "tainan_library_drive_time_interactive.html"),
    ("driving", driving_df, DISTANCE, "distance_km", "km", "到最近圖書館行車距離",
     OUTPUT_DIR / "tainan_library_drive_dist_interactive.html"),
    ("walking", walking_df, WALKING, "time_min", "分鐘", "到最近圖書館步行時間",
     OUTPUT_DIR / "tainan_library_walk_time_interactive.html"),
]
for profile, df, scale, col, unit, title, path in interactive_paths:
    if df is None:
        continue
    merged = villages.merge(
        df[["village_id", col, "nearest_library", "method"]],
        on="village_id", how="left",
    )
    last_map = render_interactive_map(merged, scale, col, unit, title, path)

# Display the last map in the notebook
last_map